## AlexNet Main Base Model -2012

In [ ]:
import torch
import torch.nn as nn

In [ ]:
class AlexNet(nn.Module):
    def __init__(self, num_classes: int = 1000):
        super(AlexNet, self).__init__()

        self.features=nn.Sequential(
        # Layer 1
        nn.Conv2d(in_channels=3,out_channels=96,kernel_size=11,stride=4,padding=2),
        nn.ReLU(inplace=True),
        nn.LocalResponseNorm(size=5, alpha=1e-4, beta=0.75, k=2.0),
        nn.MaxPool2d(kernel_size=3, stride=2),

        # Layer 2
        nn.Conv2d(in_channels=96,out_channels=256,kernel_size=5,stride=1,padding=2),
        nn.ReLU(inplace=True),
        nn.LocalResponseNorm(size=5, alpha=1e-4, beta=0.75, k=2.0),
        nn.MaxPool2d(kernel_size=3, stride=2),

        # Layer 3
        nn.Conv2d(in_channels=256,out_channels=384,kernel_size=3,stride=1,padding=2),
        nn.ReLU(inplace=True),
        # nn.LocalResponseNorm(size=5, alpha=1e-4, beta=0.75, k=2.0),

        # Layer 4
        nn.Conv2d(in_channels=384,out_channels=384,kernel_size=3,stride=1,padding=2),
        nn.ReLU(inplace=True),
        # nn.LocalResponseNorm(size=5, alpha=1e-4, beta=0.75, k=2.0),

        # Layer 5
        nn.Conv2d(in_channels=384,out_channels=256,kernel_size=3,stride=1,padding=2),
        nn.ReLU(inplace=True),
        # nn.LocalResponseNorm(size=5, alpha=1e-4, beta=0.75, k=2.0),
        nn.MaxPool2d(kernel_size=3, stride=2),

        )

        # Guarantees spatial dimensions of 6x6 before fully connected layers
        self.avgpool = nn.AdaptiveAvgPool2d((6, 6))
        
        self.classifier = nn.Sequential(
            # Layer 6
            nn.Dropout(p=0.5),
            nn.Linear(256 * 6 * 6, 4096),
            nn.ReLU(inplace=True),
            
            # Layer 7
            nn.Dropout(p=0.5),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            
            # Layer 8
            nn.Linear(4096, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
            x = self.features(x)
            x = self.avgpool(x)
            x = torch.flatten(x, 1)
            x = self.classifier(x)
            return x

# Verification script
if __name__ == "__main__":
    model = AlexNet(num_classes=10)
    sample_input = torch.randn(1, 3, 224, 224)
    output = model(sample_input)
    
    print(f"Input batch shape:  {sample_input.shape}")
    print(f"Output batch shape: {output.shape}")
        
        

### Key Innovations in AlexNet

- **ReLU Activation**:While LeNet used tanh activations, AlexNet introduced ReLU, which accelerates convergence and reduces training time.
- **GPU Utilization**: AlexNet was one of the first deep learning models to leverage GPU parallelism, using two GPUs in training to handle the large model and dataset.
- **Dropout Regularization**: AlexNet introduced dropout, a regularization technique that randomly “drops” neurons during training to reduce overfitting.
- **Data Augmentation**: To further reduce overfitting, AlexNet applied techniques like random cropping and horizontal flipping, significantly expanding the effective dataset.

## VGGNet - 16 layers - 2014

VGG-16 Architecture Breakdown
- Stage 1: 2x Conv(64) $\rightarrow$ MaxPool
- Stage 2: 2x Conv(128) $\rightarrow$ MaxPool
- Stage 3: 3x Conv(256) $\rightarrow$ MaxPool
- Stage 4: 3x Conv(512) $\rightarrow$ MaxPool
- Stage 5: 3x Conv(512) $\rightarrow$ MaxPool
- Classifier: FC(4096) $\rightarrow$ ReLU $\rightarrow$ Dropout $\rightarrow$ FC(4096) $\rightarrow$ ReLU $\rightarrow$ Dropout $\rightarrow$ FC(1000) $\rightarrow$ Softmax

In [ ]:

# Configuration for VGG-16 architecture
# 'M' represents MaxPool, numbers represent output feature channels
VGG16_CONFIG = [
    64, 64, 'M',
    128, 128, 'M',
    256, 256, 256, 'M',
    512, 512, 512, 'M',
    512, 512, 512, 'M'
]

class VGG16(nn.Module):
    def __init__(self, num_classes: int = 1000):
        super(VGG16, self).__init__()
        
        self.features = self._make_layers(VGG16_CONFIG)
        self.avgpool = nn.AdaptiveAvgPool2d((7, 7))
        
        self.classifier = nn.Sequential(
            nn.Linear(512 * 7 * 7, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),
            nn.Linear(4096, num_classes),
        )

    def _make_layers(self, config: list) -> nn.Sequential:
        layers = []
        in_channels = 3
        
        for layer in config:
            if layer == 'M':
                layers.append(nn.MaxPool2d(kernel_size=2, stride=2))
            else:
                layers.extend([
                    nn.Conv2d(in_channels, layer, kernel_size=3, padding=1),
                    nn.ReLU(inplace=True)
                ])
                in_channels = layer
                
        return nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

# Verification script
if __name__ == "__main__":
    model = VGG16(num_classes=10)
    sample_input = torch.randn(1, 3, 224, 224)
    output = model(sample_input)
    
    print(f"Input batch shape:  {sample_input.shape}")
    print(f"Output batch shape: {output.shape}")

**Key Architectural Innovations**

- Stacked $3 \times 3$ Filters: Replaced large initial filters (like AlexNet's $11 \times 11$) with stacks of small $3 \times 3$ filters. Two stacked $3 \times 3$ layers cover an effective receptive field of $5 \times 5$, while three cover $7 \times 7$.
- Increased Non-Linearity: Using multiple smaller layers instead of one large layer inserts more activation functions ($\text{ReLU}$), allowing the network to learn more complex features.
- Parameter Savings: Stacking three $3 \times 3$ layers uses $3 \times (3^2 \cdot C^2) = 27C^2$ weights, compared to a single $7 \times 7$ layer which requires $1 \times (7^2 \cdot C^2) = 49C^2$ weights—a 45% reduction in parameter count for the same receptive field.
- Systematic Doubling: Feature map channels systematically double after every max-pooling layer ($64 \rightarrow 128 \rightarrow 256 \rightarrow 512$).

## ResNet 2015